In [20]:
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from statsmodels.tsa.arima.model import ARIMA
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

EXPORTERS = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Colombia', 'Mexico',
             'Brazil', 'Egypt', 'Malaysia', 'Qatar']
CONTROLS = ['Chile', 'China', 'Indonesia', 'Philippines',
            'South Africa', 'South Korea', 'Thailand', 'Turkey']

CDS_COL = 'cds_spread'

df = pd.read_csv('../output/Current_results/complete_results_weekly_15_dampening.csv')

In [18]:
def pseudo_uniform(x):
    n = len(x)
    return rankdata(x) / (n + 1)


def fit_pair(cds_vals, dd_vals):
    """Fit Gaussian, Student-t, and Gumbel to (CDS, -DD) within country.
    Joint stress (high CDS, low DD) is upper-right corner after the -DD flip,
    so Gumbel captures the relevant tail asymmetry."""
    u = pseudo_uniform(cds_vals)
    v = pseudo_uniform(-dd_vals)
    data = np.asfortranarray(np.column_stack([u, v]))

    # Gaussian
    cop_g = pv.Bicop()
    cop_g.select(data, pv.FitControlsBicop(family_set=[pv.BicopFamily.gaussian]))
    rho_g = cop_g.parameters[0][0]
    aic_g = cop_g.aic(data)

    # Student-t (symmetric tail)
    cop_t = pv.Bicop()
    cop_t.select(data, pv.FitControlsBicop(family_set=[pv.BicopFamily.student]))
    rho_t = cop_t.parameters[0][0]
    nu_t = cop_t.parameters[1][0]
    aic_t = cop_t.aic(data)

    # Gumbel (upper-tail only — captures joint stress)
    cop_gu = pv.Bicop()
    cop_gu.select(data, pv.FitControlsBicop(family_set=[pv.BicopFamily.gumbel]))
    theta_gu = cop_gu.parameters[0][0]
    lambda_U = 2 - 2 ** (1 / theta_gu) if theta_gu > 1 else 0.0
    aic_gu = cop_gu.aic(data)

    aics = {'Gaussian': aic_g, 'Student': aic_t, 'Gumbel': aic_gu}
    best = min(aics, key=aics.get)

    return {
        'Gauss_ρ':     round(rho_g, 3),
        'Gauss_AIC':   round(aic_g, 1),
        'Student_ρ':   round(rho_t, 3),
        'Student_ν':   round(nu_t, 2),
        'Student_AIC': round(aic_t, 1),
        'Gumbel_θ':    round(theta_gu, 3),
        'Gumbel_λU':   round(lambda_U, 3),
        'Gumbel_AIC':  round(aic_gu, 1),
        'Best':        best,
    }

def filter_arma(x, order=(1, 0, 0), max_p=2, max_q=2):
    """
    Fit ARMA to series x and return standardized residuals.
    Tries the specified order first; if it fails, searches small orders by AIC.
    """
    x_clean = pd.Series(x).dropna().values
    if len(x_clean) < 50:
        # Too short to filter — return demeaned, standardized
        z = (x_clean - x_clean.mean()) / x_clean.std()
        return z

    best_aic = np.inf
    best_resid = None

    # Try the specified order first
    candidate_orders = [order]
    # Then try a small grid as fallback
    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if (p, 0, q) != order and (p + q) > 0:
                candidate_orders.append((p, 0, q))

    for ord_try in candidate_orders:
        try:
            model = ARIMA(x_clean, order=ord_try).fit()
            if model.aic < best_aic:
                best_aic = model.aic
                best_resid = model.resid
        except Exception:
            continue

    if best_resid is None:
        z = (x_clean - x_clean.mean()) / x_clean.std()
    else:
        z = (best_resid - best_resid.mean()) / best_resid.std()

    return z


In [19]:
models = {'M0': 'DD_M0', 'M1': 'DD_M1', 'M2': 'DD_M2'}
countries = EXPORTERS + CONTROLS
rows = []

for country in countries:
    group = 'Exporter' if country in EXPORTERS else 'Control'
    for mname, DD_COL in models.items():
        mask = df['country'] == country
        cds = df.loc[mask, CDS_COL].dropna()
        dd = df.loc[mask, DD_COL].dropna()
        idx = cds.index.intersection(dd.index)

        res = fit_pair(cds.loc[idx].values, dd.loc[idx].values)
        res.update({'Country': country, 'Group': group, 'Model': mname})
        rows.append(res)

results = pd.DataFrame(rows)
results

# -------------------------------------------------------------------
df_filt = df.copy()
df_filt = df_filt.sort_values(['country', 'date']).reset_index(drop=True)

# We filter CDS, DD_M0, DD_M1, DD_M2 within each country
filter_cols = [CDS_COL, 'DD_M0', 'DD_M1', 'DD_M2']

for col in filter_cols:
    df_filt[f'{col}_filt'] = np.nan

for country in EXPORTERS + CONTROLS:
    mask = df_filt['country'] == country
    sub = df_filt.loc[mask].sort_values('date')

    for col in filter_cols:
        x = sub[col].values
        # Only filter where data is contiguous (drop NaNs at ends, fill internal)
        valid = ~np.isnan(x)
        if valid.sum() < 50:
            continue

        # Filter the contiguous valid block
        x_valid = x[valid]
        z = filter_arma(x_valid, order=(1, 0, 1))

        # Place residuals back; pad with NaN where we had NaN
        z_full = np.full(len(x), np.nan)
        z_full[valid] = z

        df_filt.loc[mask, f'{col}_filt'] = z_full

print('Filtering complete. Sample standardized residuals (Saudi Arabia, M0):')
print(df_filt[df_filt['country'] == 'Saudi Arabia'][[CDS_COL, f'{CDS_COL}_filt',
                                                       'DD_M0', 'DD_M0_filt']].head(10))

/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-inverti

Filtering complete. Sample standardized residuals (Saudi Arabia, M0):
      cds_spread  cds_spread_filt     DD_M0  DD_M0_filt
5220    70.34000        -1.610074  4.093540    0.303302
5221    69.34999        -0.175802  4.093232    0.012802
5222    68.37000        -0.177854  4.093581    0.009856
5223    68.37000        -0.051333  4.091041    0.003274
5224    68.35999        -0.052660  4.080290   -0.016405
5225    72.32999         0.474733  4.082242    0.000678
5226    74.29999         0.223251  4.082558    0.004925
5227    74.29999        -0.031085  4.082526    0.006099
5228    74.29999        -0.031085  4.000218   -0.161804
5229    73.29999        -0.163604  4.000203   -0.069391


/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [21]:
summary_indiv = (results.groupby(['Group', 'Model'])
                 .agg(
                     mean_ρ=('Student_ρ', 'mean'),
                     mean_ν=('Student_ν', 'mean'),
                     median_ν=('Student_ν', 'median'),
                     mean_λ=('Student_λ', 'mean'),
                     student_pct=('Best', lambda x: f"{(x == 'Student').mean():.0%}"),
                 )
                 .round(3))

print("\nIndividual-country Student-t fits, summarised by group × model")
print("=" * 70)
print(summary_indiv.to_string())

KeyError: "Label(s) ['Student_λ'] do not exist"

In [22]:
rows = []
for group_name, group_countries in [('Exporter', EXPORTERS), ('Control', CONTROLS)]:
    for mname, DD_COL in models.items():
        u_pool, v_pool = [], []
        for country in group_countries:
            mask = df['country'] == country
            cds = df.loc[mask, CDS_COL].dropna()
            dd = df.loc[mask, DD_COL].dropna()
            idx = cds.index.intersection(dd.index)
            u_pool.append(pseudo_uniform(cds.loc[idx].values))
            v_pool.append(pseudo_uniform(-dd.loc[idx].values))

        u = np.concatenate(u_pool)
        v = np.concatenate(v_pool)
        res = fit_pair_from_pseudouniforms(u, v)
        res.update({'Group': group_name, 'Model': mname})
        rows.append(res)

pooled_within = pd.DataFrame(rows)[['Group', 'Model',
                                     'Gauss_ρ', 'Gauss_AIC',
                                     'Student_ρ', 'Student_ν', 'Student_λ', 'Student_AIC',
                                     'Best']]
print("\nPooled fits on within-country pseudo-uniforms")
print("=" * 70)
pooled_within


Pooled fits on within-country pseudo-uniforms


,Group,Model,Gauss_ρ,Gauss_AIC,Student_ρ,Student_ν,Student_λ,Student_AIC,Best
0,Exporter,M0,0.429,-819.5,0.453,10.16,0.065,-874.2,Student
1,Exporter,M1,0.514,-1244.3,0.531,7.96,0.132,-1309.2,Student
2,Exporter,M2,0.571,-1605.4,0.586,10.56,0.109,-1657.4,Student
3,Control,M0,0.114,-50.2,0.113,50.00,0.000,-45.8,Gaussian
4,Control,M1,0.215,-187.1,0.217,9.19,0.028,-214.9,Student
5,Control,M2,0.316,-421.4,0.313,18.82,0.004,-428.6,Student


In [12]:
rows = []
for group_name, group_countries in [('Exporter', EXPORTERS), ('Control', CONTROLS)]:
    df_g = df[df['country'].isin(group_countries)]
    for mname, DD_COL in models.items():
        cds = df_g[CDS_COL].dropna()
        dd = df_g[DD_COL].dropna()
        idx = cds.index.intersection(dd.index)

        # Pool first, then ECDF — the "old" approach
        res = fit_pair(cds.loc[idx].values, dd.loc[idx].values)
        res.update({'Group': group_name, 'Model': mname})
        rows.append(res)

pooled_old = pd.DataFrame(rows)[['Group', 'Model',
                                  'Gauss_ρ', 'Gauss_AIC',
                                  'Student_ρ', 'Student_ν', 'Student_λ', 'Student_AIC',
                                  'Best']]
print("\nPooled fits with pool-first-then-ECDF (the old approach)")
print("=" * 70)
pooled_old


Pooled fits with pool-first-then-ECDF (the old approach)


,Group,Model,Gauss_ρ,Gauss_AIC,Student_ρ,Student_ν,Student_λ,Student_AIC,Best
0,Exporter,M0,0.196,-160.6,0.189,9.70,0.021,-182.2,Student
1,Exporter,M1,0.162,-108.5,0.154,8.86,0.023,-135.3,Student
2,Exporter,M2,-0.122,-60.0,-0.138,5.92,0.020,-143.5,Student
3,Control,M0,0.554,-1520.3,0.559,15.19,0.048,-1535.0,Student
4,Control,M1,0.567,-1613.1,0.569,29.85,0.007,-1614.6,Student
5,Control,M2,0.616,-1984.6,0.617,50.00,0.001,-1982.8,Gaussian


In [13]:
print("\nSIDE BY SIDE: ν and ρ under three estimation approaches")
print("=" * 90)

merged = (
    summary_indiv.reset_index()[['Group', 'Model', 'mean_ν', 'mean_ρ']]
    .rename(columns={'mean_ν': 'ν_indiv_mean', 'mean_ρ': 'ρ_indiv_mean'})
    .merge(
        pooled_within[['Group', 'Model', 'Student_ν', 'Student_ρ']]
        .rename(columns={'Student_ν': 'ν_pooled_within', 'Student_ρ': 'ρ_pooled_within'}),
        on=['Group', 'Model']
    )
    .merge(
        pooled_old[['Group', 'Model', 'Student_ν', 'Student_ρ']]
        .rename(columns={'Student_ν': 'ν_pooled_old', 'Student_ρ': 'ρ_pooled_old'}),
        on=['Group', 'Model']
    )
)
print(merged.to_string(index=False))


SIDE BY SIDE: ν and ρ under three estimation approaches
   Group Model  ν_indiv_mean  ρ_indiv_mean  ν_pooled_within  ρ_pooled_within  ν_pooled_old  ρ_pooled_old
 Control    M0        35.488         0.116            50.00            0.113         15.19         0.559
 Control    M1        28.920         0.207             9.19            0.217         29.85         0.569
 Control    M2        36.275         0.308            18.82            0.313         50.00         0.617
Exporter    M0        28.984         0.446            10.16            0.453          9.70         0.189
Exporter    M1        17.868         0.504             7.96            0.531          8.86         0.154
Exporter    M2        25.196         0.582            10.56            0.586          5.92        -0.138


In [23]:
# -------------------------------------------------------------------
# Re-fit using ARMA-filtered residuals
# -------------------------------------------------------------------
CDS_COL_F = f'{CDS_COL}_filt'
models_filt = {'M0': 'DD_M0_filt', 'M1': 'DD_M1_filt', 'M2': 'DD_M2_filt'}

rows_filt = []
for country in EXPORTERS + CONTROLS:
    group = 'Exporter' if country in EXPORTERS else 'Control'
    for mname, DD_COL in models_filt.items():
        mask = df_filt['country'] == country
        cds = df_filt.loc[mask, CDS_COL_F].dropna()
        dd  = df_filt.loc[mask, DD_COL].dropna()
        idx = cds.index.intersection(dd.index)

        if len(idx) < 50:
            continue

        res = fit_pair(cds.loc[idx].values, dd.loc[idx].values)
        res.update({'Country': country, 'Group': group, 'Model': mname})
        rows_filt.append(res)

results_filt = pd.DataFrame(rows_filt)
cols = ['Country', 'Group', 'Model',
        'Student_ρ', 'Student_ν', 'Student_AIC',
        'Gumbel_θ', 'Gumbel_λU', 'Gumbel_AIC',
        'ΔAIC', 'Best']
# Some columns may not exist depending on which fit_pair you have — adjust accordingly
results_filt = results_filt[[c for c in cols if c in results_filt.columns]]
results_filt

,Country,Group,Model,Student_ρ,Student_ν,Student_AIC,Gumbel_θ,Gumbel_λU,Gumbel_AIC,Best
0,Saudi Arabia,Exporter,M0,0.062,50.00,3.8,1.017,0.023,1.6,Gaussian
1,Saudi Arabia,Exporter,M1,0.062,50.00,2.0,1.040,0.053,-1.1,Gumbel
2,Saudi Arabia,Exporter,M2,0.218,31.17,-18.5,1.144,0.167,-20.7,Gaussian
3,UAE (Abu Dhabi),Exporter,M0,0.023,6.21,-6.7,1.036,0.047,-0.4,Student
4,UAE (Abu Dhabi),Exporter,M1,0.059,5.76,-8.9,1.067,0.085,-7.2,Student
5,UAE (Abu Dhabi),Exporter,M2,0.239,7.64,-34.4,1.185,0.205,-37.4,Gumbel
6,Colombia,Exporter,M0,0.302,11.03,-45.0,1.253,0.261,-61.4,Gumbel
7,Colombia,Exporter,M1,0.216,6.16,-29.1,1.179,0.199,-36.3,Gumbel
8,Colombia,Exporter,M2,0.452,50.00,-110.5,1.410,0.365,-120.9,Gumbel
9,Mexico,Exporter,M0,0.119,9.29,-8.7,1.105,0.128,-18.5,Gumbel


In [24]:
# -------------------------------------------------------------------
# Compare unfiltered vs filtered: best-family counts by group × model
# -------------------------------------------------------------------
print('UNFILTERED — best-family counts (out of 8 countries per cell)')
print('=' * 70)
print(results.groupby(['Group', 'Model', 'Best']).size().unstack(fill_value=0))

print()
print('ARMA-FILTERED — best-family counts (out of 8 countries per cell)')
print('=' * 70)
print(results_filt.groupby(['Group', 'Model', 'Best']).size().unstack(fill_value=0))

# Per-country side-by-side for the commodity-exposed controls
focus = ['Chile', 'Indonesia', 'South Africa', 'Saudi Arabia', 'Mexico']
print()
print('PER-COUNTRY COMPARISON — Best family before/after filtering')
print('=' * 70)
comp = (results[results['Country'].isin(focus)][['Country', 'Model', 'Best']]
        .rename(columns={'Best': 'Best_unfiltered'})
        .merge(
            results_filt[results_filt['Country'].isin(focus)][['Country', 'Model', 'Best']]
            .rename(columns={'Best': 'Best_filtered'}),
            on=['Country', 'Model']
        ))
print(comp.to_string(index=False))

UNFILTERED — best-family counts (out of 8 countries per cell)
Best            Gaussian  Gumbel  Student
Group    Model                           
Control  M0            5       3        0
         M1            2       5        1
         M2            4       3        1
Exporter M0            4       1        3
         M1            1       6        1
         M2            1       5        2

ARMA-FILTERED — best-family counts (out of 8 countries per cell)
Best            Gaussian  Gumbel  Student
Group    Model                           
Control  M0            1       5        2
         M1            0       2        6
         M2            1       7        0
Exporter M0            1       6        1
         M1            0       7        1
         M2            2       6        0

PER-COUNTRY COMPARISON — Best family before/after filtering
     Country Model Best_unfiltered Best_filtered
Saudi Arabia    M0        Gaussian      Gaussian
Saudi Arabia    M1        Gaussian       